# Logistic Regression Analysis – Airline Customer Satisfaction Prediction

This notebook follows all project requirements:
- Data inspection and preprocessing
- Encoding categorical variables
- Train/Test split
- Logistic Regression model
- Confusion Matrix, Precision, Recall, Accuracy
- Coefficient interpretation
- Business recommendations
- Limitations and next steps


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score, classification_report

df = pd.read_csv('invistico_Airline.csv')
df.head()


,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,...,4,2,2,0,2,4,2,5,0,0.0


## Dataset Inspection

In [2]:
print('Dataset Shape:', df.shape)
print('\nTarget Distribution')
print(df['satisfaction'].value_counts())
print('\nMissing Values')
print(df.isnull().sum().sort_values(ascending=False).head())


Dataset Shape: (129880, 22)

Target Distribution
satisfaction
satisfied       71087
dissatisfied    58793
Name: count, dtype: int64

Missing Values
Arrival Delay in Minutes      393
Customer Type                   0
Departure Delay in Minutes      0
Online boarding                 0
Cleanliness                     0
dtype: int64


## Feature Engineering and Encoding

In [3]:
X = df.drop('satisfaction', axis=1)
y = (df['satisfaction'] == 'satisfied').astype(int)

categorical_features = X.select_dtypes(include='object').columns.tolist()
numerical_features = [c for c in X.columns if c not in categorical_features]

preprocessor = ColumnTransformer([
    ('num',
     Pipeline([
         ('imputer', SimpleImputer(strategy='median')),
         ('scaler', StandardScaler())
     ]),
     numerical_features),
    ('cat',
     Pipeline([
         ('imputer', SimpleImputer(strategy='most_frequent')),
         ('encoder', OneHotEncoder(handle_unknown='ignore'))
     ]),
     categorical_features)
])


## Train/Test Split and Model Training

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Model Evaluation

In [5]:
predictions = model.predict(X_test)

cm = confusion_matrix(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
accuracy = accuracy_score(y_test, predictions)

print('Confusion Matrix\n', cm)
print('\nAccuracy:', round(accuracy,4))
print('Precision:', round(precision,4))
print('Recall:', round(recall,4))

print('\nClassification Report')
print(classification_report(y_test, predictions))


Confusion Matrix
 [[ 9544  2215]
 [ 2233 11984]]

Accuracy: 0.8288
Precision: 0.844
Recall: 0.8429

Classification Report
              precision    recall  f1-score   support

           0       0.81      0.81      0.81     11759
           1       0.84      0.84      0.84     14217

    accuracy                           0.83     25976
   macro avg       0.83      0.83      0.83     25976
weighted avg       0.83      0.83      0.83     25976



### Interpretation of Results

Observed results:

- Accuracy ≈ 82.88%
- Precision ≈ 84.40%
- Recall ≈ 84.29%

Confusion Matrix:

| Actual / Predicted | Dissatisfied | Satisfied |
|---|---:|---:|
| Dissatisfied | 9544 | 2215 |
| Satisfied | 2233 | 11984 |

The model performs well and maintains a strong balance between Precision and Recall.


## Model Coefficients

In [6]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)

coef_df[['Feature','Coefficient']].head(15)


,Feature,Coefficient
19,cat__Customer Type_disloyal Customer,-1.046129
7,num__Inflight entertainment,0.972683
18,cat__Customer Type_Loyal Customer,0.826756
21,cat__Type of Travel_Personal Travel,-0.486751
22,cat__Class_Business,0.423151
2,num__Seat comfort,0.392571
10,num__On-board service,0.388175
13,num__Checkin service,0.354962
24,cat__Class_Eco Plus,-0.349142
9,num__Ease of Online booking,0.333694


## Key Drivers of Satisfaction

Strong positive drivers:
- Inflight entertainment
- Seat comfort
- On-board service
- Check-in service
- Ease of online booking
- Leg room service
- Business class travel
- Loyal customers

Strong negative drivers:
- Disloyal customer status
- Personal travel
- Economy/Eco Plus class
- Poor departure/arrival convenience
- Lower food and drink ratings

These variables have the largest influence on the probability of a passenger being satisfied.


## Business Recommendations

1. Improve Inflight Wi‑Fi and Digital Services
   - Invest in better connectivity and online support systems.

2. Increase Entertainment Quality
   - Entertainment is among the strongest satisfaction drivers.

3. Focus on Loyalty Programs
   - Loyal customers show substantially higher satisfaction probabilities.

4. Improve Economy-Class Experience
   - Seat comfort, leg room, and service quality can reduce dissatisfaction.

5. Strengthen Online Booking and Check‑in Processes
   - Streamlined digital experiences positively influence satisfaction.

6. Target Business Travelers
   - Business travelers tend to report higher satisfaction and represent a valuable customer segment.


## Limitations and Next Steps

Limitations:
- Logistic Regression assumes linear relationships in log-odds.
- Feature interactions are not explicitly modeled.
- Satisfaction may be influenced by variables not present in the dataset.

Next Steps:
- Compare performance against Random Forest, XGBoost, and Gradient Boosting.
- Perform hyperparameter tuning.
- Conduct feature importance analysis and SHAP interpretation.
- Deploy the model through an API or dashboard for operational use.
